# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv

import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from pydantic import BaseModel, Field

In [3]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")

MODEL="gpt-4o-mini"

In [4]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=OPENAI_BASE_URL,
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)
    return [
        {
            "Platform": m.get("Platform"),
            "Name": m.get("Name"),
            "YearOfRelease": m.get("YearOfRelease"),
            "Description": m.get("Description"),
        }
        for m in results["metadatas"][0]
    ]

#### Evaluate Retrieval Tool

In [6]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are enough to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation")

judge = LLM(model=MODEL)

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Documents: {retrieved_docs}"
    )
    response = judge.invoke(prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(response.content)
    return report.model_dump()

#### Game Web Search Tool

In [7]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> list:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    response = tavily_client.search(query=question, max_results=3)
    return [
        {
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
        }
        for r in response["results"]
    ]

### Agent

In [8]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

instructions = """You are UdaPlay, an AI assistant that answers questions about video games.

Follow this process for every new question:
1. Call retrieve_game to search the internal vector database.
2. Call evaluate_retrieval with the original question and the retrieved documents.
3. If useful is true, answer using only the retrieved documents.
4. If useful is false, call game_web_search and answer from the web results.
   Always cite the source URL(s) you used.

Rules:
- Before each tool call, briefly explain why you are calling it.
- If the answer is already in the conversation history, answer from it without calling tools.
- Always retrieve first and always evaluate before deciding to search the web.
- Match both the game and the platform in the question.
- When comparing or ordering games by date, first list each game with its year, sorted from earliest to latest, and only then state which came first.
- State clearly whether the answer came from the internal database or the web.
- If neither source answers the question, say so instead of guessing.
"""

agent = Agent(
    model_name=MODEL,
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

In [9]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

def run_query(query: str, session_id: str):
    print(f"\n{'=' * 80}\nQUERY: {query}\n{'=' * 80}")
    run = agent.invoke(query, session_id=session_id)
    messages = run.get_final_state()["messages"]
    start = max(i for i, m in enumerate(messages) if m.role == "user")
    for m in messages[start:]:
        if m.role == "assistant" and m.tool_calls:
            if m.content:
                print(f"[REASONING] {m.content}")
            for c in m.tool_calls:
                print(f"[TOOL CALL] {c.function.name}({c.function.arguments[:200]})")
        elif m.role == "tool":
            limit = 800 if m.name == "evaluate_retrieval" else 300
            print(f"[TOOL RESULT: {m.name}] {m.content[:limit]}")
        elif m.role == "assistant":
            print(f"\n[FINAL ANSWER]\n{m.content}")

SESSION = "udaplay-demo"

run_query("When was Pokémon Gold and Silver released?", SESSION)
run_query("Which one was the first 3D platformer Mario game?", SESSION)
run_query("Was Mortal Kombat X released for PlayStation 5?", SESSION)

# test memory
run_query("Which of the games we just discussed was released first?", SESSION) 


QUERY: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[TOOL CALL] retrieve_game({"query":"Pokémon Gold and Silver release date"})
[TOOL RESULT: retrieve_game] "[{'Platform': 'Game Boy Color', 'Name': 'Pok\u00e9mon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pok\u00e9mon Ruby and Sapphire', 'YearOfRel
[TOOL CALL] evaluate_retrieval({"question":"When was Pokémon Gold and Silver released?","retrieved_docs":"[{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation P)
[TOOL RESULT: evaluate_retrieval] "{'useful': True, 'description': 'The documents provided contain sufficient information to answer the q

### (Optional) Advanced

In [10]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes